# Figure 4 | Utah FORGE modelling and gravity assessment

Run the cells in order from the repository root or `examples/`. Panels are saved to `examples/Figure/fig4/` at 400 dpi. The source field data and saved case results must be obtained separately. The assembled manuscript figure is not generated here.

The gravity assessment is calculated by the case workflow, not by the `ofag.audit` tool. The saved geological model predates the retrospective review.

## Setup and shared map frame

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.tri as mtri
from matplotlib.colors import LightSource, LogNorm, Normalize, TwoSlopeNorm
from matplotlib.cm import ScalarMappable
from matplotlib.ticker import MaxNLocator
from matplotlib.patches import Patch
from mpl_toolkits.axes_grid1 import make_axes_locatable
from mpl_toolkits.mplot3d.art3d import Line3DCollection
from scipy.interpolate import griddata
from scipy.spatial import cKDTree
from PIL import Image

def find_project_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / 'examples').is_dir() and (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('Run this notebook from the project root or examples directory.')


ROOT = find_project_root()
OUT = ROOT / 'examples' / 'Figure' / 'fig4'
OUT.mkdir(parents=True, exist_ok=True)

STATE = json.loads((ROOT / 'Result' / 'Utah-FORGE' / 'case_state.json').read_text(encoding='utf-8'))
RUNS = ROOT / 'Result' / 'Utah-FORGE' / 'runs'
GRAVITY_RUN = RUNS / STATE['run_id']
TEM_RUN = RUNS / STATE['tem_run_id']
MT_RUN = RUNS / STATE['mt_run_id']
IMPORTS = ROOT / 'Result' / 'Utah-FORGE' / 'imports'
GEO_MODEL = IMPORTS / 'geological_model.npz'
MODEL_EXTENT_M = (331_900.0, 340_150.0, 4_259_925.0, 4_265_150.0)
ELEVATION_RANGE_M = (-1_200.0, 2_490.0)

mpl.rcParams.update({
    'font.family': 'Arial', 'font.size': 11, 'axes.linewidth': 1.1,
    'axes.labelsize': 14, 'xtick.labelsize': 11, 'ytick.labelsize': 11,
    'axes.grid': False, 'legend.frameon': False,
    'savefig.facecolor': 'white', 'figure.facecolor': 'white',
})
RES_CMAP = mpl.colormaps['turbo'].copy(); RES_CMAP.set_bad('white')
RES_NORM = LogNorm(1.0, 1.0e4)

def bold_ticks(ax):
    ax.xaxis.label.set_fontweight('bold'); ax.yaxis.label.set_fontweight('bold')
    for tick in ax.get_xticklabels() + ax.get_yticklabels():
        tick.set_fontweight('bold')
    ax.tick_params(width=1.2, length=5)

def reduce_xy_ticks(ax, nbins=4):
    ax.xaxis.set_major_locator(MaxNLocator(nbins=nbins, min_n_ticks=3))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=nbins, min_n_ticks=3))

def add_resistivity_colorbar(fig, ax, mappable, pad=0.14):
    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='4.5%', pad=pad)
    cb = fig.colorbar(mappable, cax=cax)
    cb.set_ticks([1, 10, 100, 1000, 10000])
    cb.set_ticklabels([r'$10^0$', r'$10^1$', r'$10^2$', r'$10^3$', r'$10^4$'])
    cb.set_label(r'Resistivity ($\Omega$ m)', fontsize=14, fontweight='bold')
    cb.ax.tick_params(labelsize=11, width=1.1)
    for tick in cb.ax.get_yticklabels(): tick.set_fontweight('bold')
    return cb

# Fixed physical plot-frame geometry for panels b--d.  The colorbar in d is extra.
SECTION_FRAME_W_IN = 6.40
SECTION_FRAME_H_IN = 2.16
SECTION_LEFT_IN = 1.12
SECTION_BOTTOM_IN = 0.86
SECTION_TOP_IN = 0.24
SECTION_RIGHT_IN = 0.22
SECTION_CBAR_GAP_IN = 0.20
SECTION_CBAR_W_IN = 0.29
SECTION_CBAR_RIGHT_IN = 1.18

def section_canvas(with_colorbar=False):
    extra = (SECTION_CBAR_GAP_IN + SECTION_CBAR_W_IN + SECTION_CBAR_RIGHT_IN) if with_colorbar else SECTION_RIGHT_IN
    width = SECTION_LEFT_IN + SECTION_FRAME_W_IN + extra
    height = SECTION_BOTTOM_IN + SECTION_FRAME_H_IN + SECTION_TOP_IN
    fig = plt.figure(figsize=(width, height), facecolor='white')
    ax = fig.add_axes([
        SECTION_LEFT_IN / width, SECTION_BOTTOM_IN / height,
        SECTION_FRAME_W_IN / width, SECTION_FRAME_H_IN / height,
    ])
    cax = None
    if with_colorbar:
        cbar_left = SECTION_LEFT_IN + SECTION_FRAME_W_IN + SECTION_CBAR_GAP_IN
        cax = fig.add_axes([
            cbar_left / width, SECTION_BOTTOM_IN / height,
            SECTION_CBAR_W_IN / width, SECTION_FRAME_H_IN / height,
        ])
    return fig, ax, cax

def finish_resistivity_colorbar(fig, cax, mappable):
    cb = fig.colorbar(mappable, cax=cax)
    cb.set_ticks([1, 10, 100, 1000, 10000])
    cb.set_ticklabels([r'$10^0$', r'$10^1$', r'$10^2$', r'$10^3$', r'$10^4$'])
    cb.set_label(r'Resistivity ($\Omega$ m)', fontsize=14, fontweight='bold')
    cb.ax.tick_params(labelsize=11, width=1.2, length=5)
    for tick in cb.ax.get_yticklabels():
        tick.set_fontweight('bold')
    return cb

def save_fixed_panel(fig, filename, dpi=400):
    path = OUT / filename
    fig.savefig(path, dpi=dpi, facecolor='white')
    # Optional vector copy for rendered-text QA; off unless the variable is set.
    qa_dir = __import__('os').environ.get('FIG4_QA_PDF_DIR')
    if qa_dir:
        fig.savefig(Path(qa_dir) / Path(filename).with_suffix('.pdf'), facecolor='white')
    plt.close(fig)
    return path

def save_png(fig, filename, dpi=400):
    path = OUT / filename
    fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return path

In [ ]:
km = lambda values: np.asarray(values, float) / 1000.0
MAP_FRAME_W_IN = SECTION_FRAME_W_IN
MAP_FRAME_H_IN = MAP_FRAME_W_IN * (MODEL_EXTENT_M[3] - MODEL_EXTENT_M[2]) / (MODEL_EXTENT_M[1] - MODEL_EXTENT_M[0])


## a | Seismic basement prior

In [ ]:
granite = np.loadtxt(ROOT / STATE['granite_path'], delimiter=',', skiprows=1)
pad = 300.0
near = ((granite[:, 0] >= MODEL_EXTENT_M[0] - pad) & (granite[:, 0] <= MODEL_EXTENT_M[1] + pad)
        & (granite[:, 1] >= MODEL_EXTENT_M[2] - pad) & (granite[:, 1] <= MODEL_EXTENT_M[3] + pad))
gx = np.arange(MODEL_EXTENT_M[0], MODEL_EXTENT_M[1] + 1.0, 50.0)
gy = np.arange(MODEL_EXTENT_M[2], MODEL_EXTENT_M[3] + 1.0, 50.0)
GX, GY = np.meshgrid(gx, gy)
top_of_granite = griddata(granite[near, :2], granite[near, 2], (GX, GY), method='linear')
print(f'top of granite in the model extent: {np.nanmin(top_of_granite):.0f} to {np.nanmax(top_of_granite):.0f} m elevation')

fig, ax, cax = section_canvas(with_colorbar=True)
fig.set_size_inches(fig.get_size_inches()[0], SECTION_BOTTOM_IN + MAP_FRAME_H_IN + SECTION_TOP_IN)
w_in, h_in = fig.get_size_inches()
ax.set_position([SECTION_LEFT_IN / w_in, SECTION_BOTTOM_IN / h_in, MAP_FRAME_W_IN / w_in, MAP_FRAME_H_IN / h_in])
cax.set_position([(SECTION_LEFT_IN + MAP_FRAME_W_IN + SECTION_CBAR_GAP_IN) / w_in, SECTION_BOTTOM_IN / h_in,
                  SECTION_CBAR_W_IN / w_in, MAP_FRAME_H_IN / h_in])
surface_norm = Normalize(-1500.0, 2000.0)
image = ax.pcolormesh(km(gx), km(gy), top_of_granite, cmap='cividis', norm=surface_norm,
                      shading='nearest', rasterized=True)
levels = np.arange(-1500.0, 2001.0, 250.0)
lines = ax.contour(km(gx), km(gy), top_of_granite, levels=levels, colors='#1A1A1A', linewidths=0.55, alpha=0.75)
ax.clabel(lines, levels=[-1000.0, 0.0, 1000.0], fmt='%d', fontsize=8.5, inline=True)
ax.set_xlim(km(MODEL_EXTENT_M[0]), km(MODEL_EXTENT_M[1]))
ax.set_ylim(km(MODEL_EXTENT_M[2]), km(MODEL_EXTENT_M[3]))
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('Easting (km)'); ax.set_ylabel('Northing (km)')
reduce_xy_ticks(ax)
cb = fig.colorbar(image, cax=cax)
cb.set_ticks([-1500, -500, 500, 1500])
cb.set_label('Top of granite elevation (m)', fontsize=13, fontweight='bold')
cb.ax.tick_params(labelsize=11, width=1.2, length=5)
for tick in cb.ax.get_yticklabels():
    tick.set_fontweight('bold')
bold_ticks(ax)
save_fixed_panel(fig, 'Fig4a_seismic_basement.png')

## b | TEM resistivity section

In [ ]:
tem = np.load(TEM_RUN / 'stitched_conductivity_section.npz')
tem_xy = tem['receiver_locations_m'][:, :2].astype(float)
tem_depth = tem['layer_top_depth_m'].astype(float)
tem_sigma = tem['conductivity_s_m'].astype(float)
tem_rho = np.divide(1.0, tem_sigma, out=np.full_like(tem_sigma, np.nan), where=tem_sigma > 0)

line_mask = (tem_xy[:, 0] >= 332_500.0) & (tem_xy[:, 0] <= 337_800.0) & (tem_xy[:, 1] >= 4_262_400.0)
ordered = np.flatnonzero(line_mask)
ordered = ordered[np.argsort(tem_xy[ordered, 0])]
clusters = []
for index in ordered:
    if not clusters or np.linalg.norm(tem_xy[index] - np.median(tem_xy[clusters[-1]], axis=0)) > 100.0:
        clusters.append([index])
    else:
        clusters[-1].append(index)
line_xy, line_rho = [], []
for members in clusters:
    members = np.asarray(members, dtype=int)
    line_xy.append(np.median(tem_xy[members], axis=0))
    line_rho.append(np.nanmedian(tem_rho[members], axis=0))
line_xy, line_rho = np.asarray(line_xy), np.asarray(line_rho)
order = np.argsort(line_xy[:, 0]); line_xy, line_rho = line_xy[order], line_rho[order]

terrain = np.loadtxt(IMPORTS / 'terrain_50m.csv', delimiter=',', skiprows=1)
terrain_tree = cKDTree(terrain[:, :2])
line_ground = terrain[terrain_tree.query(line_xy)[1], 2]
line_distance = np.r_[0.0, np.cumsum(np.linalg.norm(np.diff(line_xy, axis=0), axis=1))]
distance_edges = np.linspace(line_distance.min(), line_distance.max(), 241)
distance_centres = 0.5 * (distance_edges[:-1] + distance_edges[1:])
ground_edges = np.interp(distance_edges, line_distance, line_ground)
log_rho_layers = np.vstack([
    np.interp(distance_centres, line_distance, np.log10(line_rho[:, layer]))
    for layer in range(line_rho.shape[1])
])
depth_edges = np.linspace(0.0, 130.0, 132)
depth_centres = 0.5 * (depth_edges[:-1] + depth_edges[1:])
layer_for_depth = np.searchsorted(tem_depth, depth_centres, side='right') - 1
section_rho = 10.0 ** log_rho_layers[layer_for_depth]
X_edges = np.broadcast_to(distance_edges[None, :] / 1000.0, (depth_edges.size, distance_edges.size))
Z_edges = (ground_edges[None, :] - depth_edges[:, None]) / 1000.0

fig, ax, cax = section_canvas(with_colorbar=True)
image = ax.pcolormesh(X_edges, Z_edges, section_rho, shading='flat', cmap=RES_CMAP, norm=RES_NORM)
ax.plot(distance_edges / 1000.0, ground_edges / 1000.0, color='#161616', linewidth=1.2, zorder=4)
ax.scatter(line_distance / 1000.0, line_ground / 1000.0, marker='v', s=25,
           facecolor='#161616', edgecolor='white', linewidth=0.45, zorder=5)
ax.set_xlim(distance_edges[0] / 1000.0, distance_edges[-1] / 1000.0)
ax.set_ylim((ground_edges - 130.0).min() / 1000.0, (ground_edges.max() + 18.0) / 1000.0)
ax.set_xlabel('Distance (km)'); ax.set_ylabel('Elevation (km)')
reduce_xy_ticks(ax, nbins=4)
bold_ticks(ax)
finish_resistivity_colorbar(fig, cax, image)
save_fixed_panel(fig, 'Fig4b_TEM_resistivity_section.png')
print('TEM sounding clusters in section:', len(line_distance))

## c | MT resistivity models

In [ ]:
mt = np.load(MT_RUN / 'stitched_conductivity_section.npz')
x_km = mt['easting_m'].astype(float) / 1000.0
y_km = mt['northing_m'].astype(float) / 1000.0
surface_km = mt['elevation_m'].astype(float) / 1000.0
depth_top_m = mt['layer_top_depth_m'].astype(float)
conductivity = mt['conductivity_s_m'].astype(float)
resistivity = np.divide(1.0, conductivity, out=np.full_like(conductivity, np.nan), where=conductivity > 0)
xmin, xmax, ymin, ymax = np.asarray(MODEL_EXTENT_M) / 1000.0
inside = (x_km >= xmin) & (x_km <= xmax) & (y_km >= ymin) & (y_km <= ymax)
px, py, pz, prho = x_km[inside], y_km[inside], surface_km[inside], resistivity[inside]
MAX_DEPTH_M = 5_000.0
depth_bottom_m = np.minimum(np.r_[depth_top_m[1:], np.inf], MAX_DEPTH_M)
layers = np.flatnonzero((depth_top_m < MAX_DEPTH_M) & (depth_bottom_m > depth_top_m))
segments, segment_rho = [], []
for site in range(len(px)):
    for layer in layers:
        segments.append([(px[site], py[site], pz[site] - depth_top_m[layer] / 1000.0),
                         (px[site], py[site], pz[site] - depth_bottom_m[layer] / 1000.0)])
        segment_rho.append(prho[site, layer])
segment_rho = np.asarray(segment_rho)
segment_colours = RES_CMAP(RES_NORM(np.clip(segment_rho, 1.0, 1.0e4)))

fig = plt.figure(figsize=(9.2, 6.9))
ax = fig.add_subplot(111, projection='3d'); ax.set_proj_type('ortho')
tri = mtri.Triangulation(px, py)
triangles = tri.triangles
edge_lengths = np.stack([
    np.hypot(px[triangles[:, 0]] - px[triangles[:, 1]], py[triangles[:, 0]] - py[triangles[:, 1]]),
    np.hypot(px[triangles[:, 1]] - px[triangles[:, 2]], py[triangles[:, 1]] - py[triangles[:, 2]]),
    np.hypot(px[triangles[:, 2]] - px[triangles[:, 0]], py[triangles[:, 2]] - py[triangles[:, 0]]),
])
tri.set_mask(edge_lengths.max(axis=0) > 4.0)
ax.plot_trisurf(tri, pz, color='#D9D9D6', alpha=0.32, linewidth=0, shade=True)
ax.add_collection3d(Line3DCollection(segments, colors=segment_colours, linewidths=3.1, alpha=0.96))
ax.scatter(px, py, pz + 0.035, s=18, c='#202020', depthshade=False, zorder=8)
box_z = float(np.nanmean(pz)) + 0.12
box = np.array([[xmin,ymin,box_z],[xmax,ymin,box_z],[xmax,ymax,box_z],[xmin,ymax,box_z],[xmin,ymin,box_z]])
ax.plot(box[:,0], box[:,1], box[:,2], color='#111111', linewidth=2.2, linestyle=(0,(4,2)))
ax.set(xlim=(xmin-.25,xmax+.25), ylim=(ymin-.25,ymax+.25),
       zlim=(pz.min()-MAX_DEPTH_M/1000.0,pz.max()+.35),
       xlabel='Easting (km)', ylabel='Northing (km)', zlabel='')
ax.view_init(elev=24, azim=-56); ax.set_box_aspect((1.25,.82,.62)); ax.grid(False)
# Keep the elevation axis on the visible left edge of the 3-D box.
ax.zaxis._axinfo['juggled'] = (1, 2, 0)
ax.text2D(-.035, .50, 'Elevation (km)', transform=ax.transAxes, rotation=90, va='center', ha='center', fontsize=14, fontweight='bold')
ax.tick_params(axis='z', pad=2)
reduce_xy_ticks(ax, nbins=4)
for axis in (ax.xaxis,ax.yaxis,ax.zaxis):
    axis.pane.set_facecolor((1,1,1,0)); axis.pane.set_edgecolor('#B7B7B7')
for label in (ax.xaxis.label,ax.yaxis.label,ax.zaxis.label): label.set_fontweight('bold')
for tick in ax.get_xticklabels()+ax.get_yticklabels()+ax.get_zticklabels(): tick.set_fontweight('bold')
sm = ScalarMappable(norm=RES_NORM,cmap=RES_CMAP); sm.set_array([])
cax = fig.add_axes([.865,.20,.034,.58]); cb = fig.colorbar(sm,cax=cax)
cb.set_ticks([1,10,100,1000,10000]); cb.set_ticklabels([r'$10^0$',r'$10^1$',r'$10^2$',r'$10^3$',r'$10^4$'])
cb.set_label(r'Resistivity ($\Omega$ m)',fontsize=14,fontweight='bold',labelpad=10)
for tick in cb.ax.get_yticklabels(): tick.set_fontweight('bold')
fig.subplots_adjust(left=.18,right=.83,bottom=.06,top=.98)
save_png(fig,'Fig4c_MT_resistivity_models.png')
print(f'MT models shown: {inside.sum()} of {len(x_km)}')

## d | Gravity density model

In [ ]:
# Panel d: density cutaway from the saved inversion mesh.
obs = pd.read_csv(ROOT / STATE['observations_path'])
topo = pd.read_csv(IMPORTS / 'terrain_50m.csv')
mesh = np.load(GRAVITY_RUN / 'mesh_geometry.npz')
centres = mesh['cell_centers_m'][mesh['active_cells'].astype(bool)]
density_model = np.load(GRAVITY_RUN / 'recovered_density_g_cc.npy')
density_tree = cKDTree(centres)

x0, y0 = np.median(obs.x_m), np.median(obs.y_m)
x = np.linspace(topo.x_m.min(), topo.x_m.max(), 100)
y = np.linspace(topo.y_m.min(), topo.y_m.max(), 100)
xx, yy = np.meshgrid(x, y)
top = griddata((topo.x_m, topo.y_m), topo.z_m, (xx, yy), method='linear')
top = np.where(np.isfinite(top), top, np.nanmedian(topo.z_m))
depth = 1500.0
depths = np.linspace(0.0, 3300.0, 72)

def nearest_density(a, b, c):
    points = np.c_[a.ravel(), b.ravel(), c.ravel()]
    return density_model[density_tree.query(points)[1]].reshape(a.shape)

def terrain_profile(a, b):
    values = griddata((topo.x_m, topo.y_m), topo.z_m, (a, b), method='linear')
    return np.where(np.isfinite(values), values, np.nanmedian(topo.z_m))

def density_surface(a, b, levels=None):
    levels = depths if levels is None else levels
    z = terrain_profile(a[0], b[0])[None, :] - levels[:, None]
    return z, nearest_density(a, b, z)

def surface_x(y_value):
    a, _ = np.meshgrid(x, depths)
    b = np.full_like(a, y_value)
    z, values = density_surface(a, b)
    return a, b, z, values

def surface_y(x_value):
    b, _ = np.meshgrid(y, depths)
    a = np.full_like(b, x_value)
    z, values = density_surface(a, b)
    return a, b, z, values

# The cutaway samples more finely than the sections, on a grid of its own.
xa = np.linspace(topo.x_m.min(), topo.x_m.max(), 240)
ya = np.linspace(topo.y_m.min(), topo.y_m.max(), 180)
xxa, yya = np.meshgrid(xa, ya)
topa = terrain_profile(xxa, yya)
depths_a = np.linspace(0.0, 3300.0, 160)
A_TICK_PT, A_LABEL_PT, A_CBAR_TICK_PT, A_CBAR_LABEL_PT = 11, 14, 20, 23
fig = plt.figure(figsize=(9.5, 7.0), facecolor='white')
ax = fig.add_axes([0.0, 0.0, 0.74, 1.0], projection='3d')
ax.set_proj_type('persp', focal_length=0.45)
ax.view_init(elev=26, azim=-58)
ax.set_box_aspect((1.6, 1.0, 0.62), zoom=1.05)
density_norm = TwoSlopeNorm(vmin=-0.22, vcenter=0.0, vmax=0.22)
density_cmap = mpl.colormaps['coolwarm']
terrain_norm = Normalize(1200.0, 3000.0)
cut = (xxa >= x0) & (yya <= y0)
surface_z = topa.copy()
surface_z[cut] = np.nan
ax.plot_surface(xxa / 1000, yya / 1000, surface_z / 1000,
                facecolors=mpl.colormaps['terrain'](terrain_norm(topa)),
                linewidth=0, antialiased=False, shade=False, rcount=len(ya), ccount=len(xa))
floor_z = (topa - depth).copy()
# One sample wider than the cut, so the floor meets the faces without a seam.
floor = (xxa >= x0 - np.diff(xa)[0]) & (yya <= y0 + np.diff(ya)[0])
floor_z[~floor] = np.nan
ax.plot_surface(xxa / 1000, yya / 1000, floor_z / 1000,
                facecolors=density_cmap(density_norm(nearest_density(xxa, yya, topa - depth))),
                linewidth=0, antialiased=False, shade=False, rcount=len(ya), ccount=len(xa))
# Faces, as (fixed axis, coordinate, positions along it, depth range).
upper = depths_a[depths_a <= depth]
lower = depths_a[depths_a >= depth]
# Each piece carries the cut coordinate itself, so neighbouring pieces meet rather than
# leaving a seam of background between them.
xw, xe = np.r_[xa[xa < x0], x0], np.r_[x0, xa[xa > x0]]
ys_, yn = np.r_[y0, ya[ya > y0]], np.r_[ya[ya < y0], y0]
faces = [
    ('y', ya.min(), xw, depths_a), ('y', ya.min(), xe, lower),
    ('x', xa.max(), ys_, depths_a), ('x', xa.max(), yn, lower),
    ('y', y0, xe, upper), ('x', x0, yn, upper),
]
for fixed, coordinate, along, levels in faces:
    a, _ = np.meshgrid(along, levels)
    if fixed == 'y':
        b = np.full_like(a, coordinate)
    else:
        b, a = a, np.full_like(a, coordinate)
    z, values = density_surface(a, b, levels)
    ax.plot_surface(a / 1000, b / 1000, z / 1000,
                    facecolors=density_cmap(density_norm(values)),
                    linewidth=0, antialiased=False, shade=False,
                    rcount=len(levels), ccount=len(along))
# The ground line of every face, white, so the cut reads as a cut.
def edge(xs, ys, below=0.0, width=1.6):
    ax.plot(xs / 1000, ys / 1000, (terrain_profile(xs, ys) - below) / 1000 + 0.01,
            color='white', linewidth=width, zorder=10)
edge(xw, np.full_like(xw, ya.min()))
edge(np.full_like(ys_, xa.max()), ys_)
edge(xe, np.full_like(xe, y0))
edge(np.full_like(yn, x0), yn)
ax.set_xlim(xa.min() / 1000, xa.max() / 1000)
ax.set_ylim(ya.min() / 1000, ya.max() / 1000)
ax.set_zlim(-1.8, 2.1)
ax.set_xticks([333, 336, 339])
ax.set_yticks([4260.0, 4262.5, 4265.0])
ax.set_zticks([-1, 0, 1, 2])
ax.yaxis.set_major_formatter(mpl.ticker.FormatStrFormatter('%.1f'))
for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
    axis.set_pane_color((1, 1, 1, 0))
    axis.pane.set_edgecolor('#1A1A1A')
    axis._axinfo['grid'].update(color='#DDDDDD', linewidth=0.6)
    axis.line.set_linewidth(0.8)
ax.tick_params(labelsize=A_TICK_PT, pad=2)
for label in ax.get_xticklabels() + ax.get_yticklabels() + ax.get_zticklabels():
    label.set_fontweight('bold')
ax.set_xlabel('Easting (km)', fontsize=A_LABEL_PT, fontweight='bold', labelpad=10)
ax.set_ylabel('Northing (km)', fontsize=A_LABEL_PT, fontweight='bold', labelpad=12)
ax.set_zlabel('Elevation (km)', fontsize=A_LABEL_PT, fontweight='bold', labelpad=6)
cax = fig.add_axes([0.875, 0.14, 0.032, 0.68])
cb = fig.colorbar(ScalarMappable(norm=density_norm, cmap=density_cmap), cax=cax)
cb.set_ticks([-0.2, -0.1, 0.0, 0.1, 0.2])
cb.ax.tick_params(labelsize=A_CBAR_TICK_PT)
cb.set_label(r'Density contrast (g cm$^{-3}$)', fontsize=A_CBAR_LABEL_PT)
save_png(fig, 'Fig4d_density_cutaway_3D.png')


## e | Geological interpretation

In [ ]:
geo = np.load(GEO_MODEL, allow_pickle=False)
mesh = np.load(GRAVITY_RUN / 'mesh_geometry.npz')
cell_centres = mesh['cell_centers_m']
unit = geo['unit'].astype(int)
valid = unit >= 0
unit_names = geo['unit_names'].astype(str)

# Regular display grid sampled from the final octree-cell classification.
nx, ny, nz = 48, 32, 42
xe = np.linspace(MODEL_EXTENT_M[0], MODEL_EXTENT_M[1], nx + 1)
ye = np.linspace(MODEL_EXTENT_M[2], MODEL_EXTENT_M[3], ny + 1)
ze = np.linspace(ELEVATION_RANGE_M[0], 2_150.0, nz + 1)
xc, yc, zc = (xe[:-1]+xe[1:])/2, (ye[:-1]+ye[1:])/2, (ze[:-1]+ze[1:])/2
XC, YC, ZC = np.meshgrid(xc, yc, zc, indexing='ij')
nearest = cKDTree(cell_centres[valid]).query(np.c_[XC.ravel(),YC.ravel(),ZC.ravel()])[1]
unit_grid = unit[valid][nearest].reshape(XC.shape)

topography = pd.read_csv(IMPORTS / 'terrain_50m.csv')
XX, YY = np.meshgrid(xc, yc, indexing='ij')
top = griddata((topography.x_m,topography.y_m),topography.z_m,(XX,YY),method='linear')
top = np.where(np.isfinite(top),top,np.nanmedian(topography.z_m))
active = ZC <= top[:,:,None]
xmid = 0.5*(MODEL_EXTENT_M[0]+MODEL_EXTENT_M[1])
ymid = 0.5*(MODEL_EXTENT_M[2]+MODEL_EXTENT_M[3])
cutaway = (XC >= xmid) & (YC <= ymid)
filled = active & ~cutaway

unit_colours = np.array(['#B15B5B','#4B9B92','#66578F','#D9B56D'])
facecolours = np.empty(unit_grid.shape+(4,),float)
for index, colour in enumerate(unit_colours):
    facecolours[unit_grid==index] = mpl.colors.to_rgba(colour,.98)
Xv,Yv,Zv = np.meshgrid(xe/1000.0,ye/1000.0,ze/1000.0,indexing='ij')

fig = plt.figure(figsize=(9.2,6.9),facecolor='white')
ax = fig.add_subplot(111,projection='3d'); ax.set_proj_type('ortho')
ax.voxels(Xv,Yv,Zv,filled,facecolors=facecolours,
          edgecolor=(1,1,1,.10),linewidth=.12,shade=True)

# Terrain surface uses the same palette and elevation limits as the Utah location map.
xt=np.linspace(MODEL_EXTENT_M[0],MODEL_EXTENT_M[1],100)
yt=np.linspace(MODEL_EXTENT_M[2],MODEL_EXTENT_M[3],70)
XT,YT=np.meshgrid(xt,yt,indexing='ij')
ZT=griddata((topography.x_m,topography.y_m),topography.z_m,(XT,YT),method='linear')
ZT=np.where(np.isfinite(ZT),ZT,np.nanmedian(topography.z_m))
terrain_cut=(XT>=xmid)&(YT<=ymid); Zplot=ZT.copy(); Zplot[terrain_cut]=np.nan
terrain_rgb=LightSource(azdeg=315,altdeg=38).shade(
    ZT,cmap=mpl.colormaps['terrain'],norm=Normalize(1200,3000),blend_mode='soft',fraction=.55)
ax.plot_surface(XT/1000,YT/1000,Zplot/1000,facecolors=terrain_rgb,
                linewidth=0,antialiased=True,shade=False,zorder=5)

# White terrain-contact lines conceal raster seams at the two cut faces.
xline=xt[xt>=xmid]
zline=griddata((topography.x_m,topography.y_m),topography.z_m,
               (xline,np.full_like(xline,ymid)),method='linear')
ax.plot(xline/1000,np.full_like(xline,ymid)/1000,zline/1000+.004,color='white',lw=2.2,zorder=20)
yline=yt[yt<=ymid]
zline=griddata((topography.x_m,topography.y_m),topography.z_m,
               (np.full_like(yline,xmid),yline),method='linear')
ax.plot(np.full_like(yline,xmid)/1000,yline/1000,zline/1000+.004,color='white',lw=2.2,zorder=20)

ax.set(xlim=(MODEL_EXTENT_M[0]/1000,MODEL_EXTENT_M[1]/1000),
       ylim=(MODEL_EXTENT_M[2]/1000,MODEL_EXTENT_M[3]/1000),
       zlim=(ELEVATION_RANGE_M[0]/1000,2.2),
       xlabel='Easting (km)',ylabel='Northing (km)',zlabel='')
ax.set_box_aspect((1.25,.82,.62)); ax.view_init(elev=24,azim=-56); ax.grid(False)
reduce_xy_ticks(ax, nbins=4)
ax.zaxis.set_major_locator(MaxNLocator(nbins=4, min_n_ticks=3))
ax.tick_params(axis='z', pad=2)
ax.text2D(1.06, .50, 'Elevation (km)', transform=ax.transAxes, rotation=90, va='center', ha='center', fontsize=14, fontweight='bold')
for axis in (ax.xaxis,ax.yaxis,ax.zaxis):
    axis.pane.fill=False; axis.pane.set_edgecolor('#B0B0B0')
for label in (ax.xaxis.label,ax.yaxis.label,ax.zaxis.label):
    label.set_fontweight('bold'); label.set_fontsize(12)
for tick in ax.get_xticklabels()+ax.get_yticklabels()+ax.get_zticklabels():
    tick.set_fontweight('bold'); tick.set_fontsize(9)
handles=[Patch(facecolor=unit_colours[i],edgecolor='none',label=unit_names[i]) for i in range(4)]
ax.legend(handles=handles,loc='upper left',bbox_to_anchor=(.01,.98),frameon=False,fontsize=10)
fig.subplots_adjust(left=.01,right=.92,bottom=.02,top=.99)
save_png(fig,'Fig4e_geological_model_3D.png')
print('Model cells:', int(valid.sum()))
print('Unit names:', ', '.join(unit_names))

## f | Conditional gravity assessment

In [ ]:
# Panel f: the gravity's verdict on the seismic surface, per 50 m column, with the
# places Witter moved that surface for his model 2.
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D

verdict_geo = np.load(GEO_MODEL, allow_pickle=False)
col_xy = verdict_geo['gravity_column_xy_m']
col_code = verdict_geo['gravity_column_verdict']
col_index = np.rint((col_xy - col_xy.min(axis=0)) / 50.0).astype(int)
vx = col_xy[:, 0].min() + 50.0 * np.arange(col_index[:, 0].max() + 1)
vy = col_xy[:, 1].min() + 50.0 * np.arange(col_index[:, 1].max() + 1)
verdict_grid = np.full((len(vy), len(vx)), np.nan)
verdict_grid[col_index[:, 1], col_index[:, 0]] = col_code
print('verdict shares:', {str(n): f'{np.mean(col_code == k):.1%}' for k, n in enumerate(verdict_geo['gravity_verdict_names'])})

# Order follows the stored codes: supports, more mass, less mass, not tested.
verdict_colours = ['#E6E1D8', '#C0392B', '#2E6DB4', '#FFFFFF']
verdict_cmap = ListedColormap(verdict_colours)
verdict_norm = BoundaryNorm(np.arange(-0.5, 4.5, 1.0), verdict_cmap.N)

original = np.loadtxt(ROOT / STATE['granite_path'], delimiter=',', skiprows=1)
modified = np.loadtxt(ROOT / STATE['granite_modified_path'], delimiter=',', skiprows=1)
VX, VY = np.meshgrid(vx, vy)
moved = (modified[cKDTree(modified[:, :2]).query(np.c_[VX.ravel(), VY.ravel()])[1], 2]
         - original[cKDTree(original[:, :2]).query(np.c_[VX.ravel(), VY.ravel()])[1], 2]).reshape(VX.shape)

fig, ax, cax = section_canvas(with_colorbar=True)
fig.set_size_inches(fig.get_size_inches()[0], SECTION_BOTTOM_IN + MAP_FRAME_H_IN + SECTION_TOP_IN)
w_in, h_in = fig.get_size_inches()
ax.set_position([SECTION_LEFT_IN / w_in, SECTION_BOTTOM_IN / h_in, MAP_FRAME_W_IN / w_in, MAP_FRAME_H_IN / h_in])
cax.set_position([(SECTION_LEFT_IN + MAP_FRAME_W_IN + SECTION_CBAR_GAP_IN) / w_in, SECTION_BOTTOM_IN / h_in,
                  SECTION_CBAR_W_IN / w_in, MAP_FRAME_H_IN / h_in])
image = ax.pcolormesh(km(vx), km(vy), verdict_grid, cmap=verdict_cmap, norm=verdict_norm,
                      shading='nearest', rasterized=True)
# The two surfaces differ by tens of metres everywhere from regridding alone; a light
# smoothing keeps that noise from drawing islands at the 50 m line.
from scipy.ndimage import gaussian_filter
moved_smooth = gaussian_filter(moved, sigma=2.0)
ax.contour(km(VX), km(VY), moved_smooth, levels=[50.0], colors='#111111', linewidths=1.3)
ax.contour(km(VX), km(VY), moved_smooth, levels=[-50.0], colors='#111111', linewidths=1.3, linestyles='--')
ax.set_xlim(km(MODEL_EXTENT_M[0]), km(MODEL_EXTENT_M[1]))
ax.set_ylim(km(MODEL_EXTENT_M[2]), km(MODEL_EXTENT_M[3]))
ax.set_aspect('equal', adjustable='box')
ax.set_xlabel('Easting (km)'); ax.set_ylabel('Northing (km)')
reduce_xy_ticks(ax)
ax.legend(handles=[Line2D([], [], color='#111111', lw=1.3, label='Witter raised'),
                   Line2D([], [], color='#111111', lw=1.3, ls='--', label='Witter lowered')],
          loc='upper left', fontsize=10, frameon=True, framealpha=0.9, edgecolor='none',
          prop={'weight': 'bold', 'size': 10})
cb = fig.colorbar(image, cax=cax, ticks=[0, 1, 2, 3])
cb.set_ticklabels(['Supports', 'More mass', 'Less mass', 'Not tested'])
cb.ax.tick_params(labelsize=11, width=1.2, length=0)
cb.outline.set_linewidth(1.1)
for tick in cb.ax.get_yticklabels():
    tick.set_fontweight('bold')
bold_ticks(ax)
save_fixed_panel(fig, 'Fig4f_gravity_assessment.png')


## Export check

Every listed panel must exist and have nonzero dimensions.

In [ ]:
from PIL import Image
expected = ['Fig4a_seismic_basement.png', 'Fig4b_TEM_resistivity_section.png', 'Fig4c_MT_resistivity_models.png', 'Fig4d_density_cutaway_3D.png', 'Fig4e_geological_model_3D.png', 'Fig4f_gravity_assessment.png']
for name in expected:
    path = OUT / name
    if not path.is_file():
        raise FileNotFoundError(path)
    with Image.open(path) as panel:
        assert panel.width > 0 and panel.height > 0, path
print(f'Figure panels checked: {len(expected)} in {OUT}')
